# NLP Group 7 Project P2

## Imports

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import json
from config import DEPLOYMENT_NAME, client
from sandbox import PersistentSolverSandbox
from agents import run_solver_with_tools
from utils import extract_answer, check_correctness, extract_json_from_response, get_llm_response
from prompts import VERIFIER_ROLE, get_solver_prompt

## Load dataset

In [ ]:
dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

## Dataset Loading and Aggregation Logic
This snippet demonstrates the initial setup for creating the unified test and evaluation pool from the various competition subsets within the MathArena benchmark.

Logic: Each subset (AIME, HMMT, etc.) is loaded separately using the load_dataset function from the datasets library and immediately converted into a pandas DataFrame. The final step in the complete notebook (though not explicitly shown here) would be to concatenate all these DataFrames into a single unified test_set_df for easy looping and evaluation. The specific note about hmmt_nov_problems lacking the problem_type field is an important data cleaning consideration for the final aggregation step.

In [ ]:
aime_problems = load_dataset("MathArena/aime_2025", split="train")
aime_df = aime_problems.to_pandas()

hmmt_feb_problems = load_dataset("MathArena/hmmt_feb_2025", split="train")
hmmt_feb_df = hmmt_feb_problems.to_pandas()

brumo_problems = load_dataset("MathArena/brumo_2025", split="train")
brumo_df = brumo_problems.to_pandas()

smt_problems = load_dataset("MathArena/smt_2025", split="train")
smt_df = smt_problems.to_pandas()

cmimc_problems = load_dataset("MathArena/cmimc_2025", split="train")
cmimc_df = cmimc_problems.to_pandas()


# Doesn't have the problem_type field
hmmt_nov_problems = load_dataset("MathArena/hmmt_nov_2025", split="train")
hmmt_nov_df = hmmt_nov_problems.to_pandas()



## General info and null values

In [ ]:
df.info()

In [ ]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

## Communication with the LLM model

This code block shows how the Azure AI environment is set up and how a direct API call is made to the DeepSeek-V3-03-24 model. This initialization is critical as it defines the endpoint for all subsequent Solver, Verifier, and Planner interactions.

## Conditional LLM Response Formatting

The get_llm_response function is the low-level wrapper for all communication with the DeepSeek-V3-0324 model. Its most critical logic is the dynamic setting of the response_format, which is essential for ensuring the Verifier's output is reliable.

## Verifier Role Definition and Output Schema (PCAF Feedback Structure)

This code block sets the system prompt ($\text{VERIFIER\_ROLE}$) for the Verifier agent and defines the strict JSON output schema it must adhere to. This structure is the fundamental communication protocol between the Verifier and the Planner.

Logic (Role): The Verifier is positioned as a Skeptical Math Auditor whose primary objective is to find flaws using a three-point checklist: Dual-Method Check, Consistency Check (text vs. code output), and Logic Trap Checks (e.g., edge cases in number theory, coordinate use in geometry).Logic (Schema): The mandatory JSON output is constrained to a specific set of $\text{error\_type}$ strings ($\text{LOGIC\_FLAW}$, $\text{VALUE\_MISMATCH}$, $\text{METHOD\_CONFLICT}$, $\text{NONE}$). These discrete, machine-readable flags are the core input for the Planner's deterministic routing logic (Section 5.3). This mechanism is designed to prevent the LLM from providing verbose, unstructured text that is difficult to parse and act upon.

## Tool Implementation: PersistentSolverSandbox

This class is the foundation of the "Code-First Mandate" in the PCAF framework, ensuring reliable, stateful, and safe code execution by the Solver.

Logic (Persistence and State): The key architectural feature is maintaining the code execution state. The self.globals dictionary holds all defined variables and pre-loaded libraries (like $\text{sympy}$ and $\text{numpy}$) across sequential calls to run_code. This allows the Solver to execute multi-step logic (e.g., define a variable in one $\text{```python```}$ block and access it in the next) via the $\text{exec(code\_str, self.globals)}$ command.

Logic (Robustness and Safety): The run_code method is wrapped by func_timeout(timeout_seconds, ...) to impose a strict 20-second time limit. This prevents infinite loops or overly complex calculations from halting the entire system. All execution outputs and errors are reliably captured using $\text{contextlib.redirect\_stdout}$ and returned to the LLM with the $\text{RUNTIME ERROR}$ tag, enabling the Solver to self-debug its code errors in the next turn.

## Execution Kernel

The run_solver_with_tools function is the operational core of the Solver agent. It manages the conversational turns within a single attempt, dynamically integrating the LLM's thought process with the results of the external PersistentSolverSandbox.

Logic (Tool Integration Loop): The function iteratively checks the LLM's response for Python code blocks using a regular expression.
-  Code Detection: If one or more code blocks are found, the conversation loop pauses the LLM's generation.
-  Execution and Feedback: Each code block is executed sequentially by the sandbox_instance.run_code(). The combined outputs including any RUNTIME_ERRORs are formatted and fed back to the LLM via a new user message tagged as $\text{OBSERVATION (Code Output)}$.
-  Trace Logging: The full conversation and the execution output are meticulously compiled into the $\text{full\_trace}$ variable, with the output marked by the $\text{[Tool Output]}$ tag. This trace is the complete evidentiary record passed to the Verifier for auditing.

Logic (Completion): The loop continues for a maximum of $\text{max\_turns}$ (3). It terminates successfully if the LLM's response contains a final answer marker (\\\boxed or $\text{Final Answer}$), indicating the Solver has reached its conclusion. If no final answer is reached after all turns, the final partial response is returned for verification.

## Few-Shot Examples for Solver Self-Correction

The FEW\_SHOT\_CORRECTION\_EXAMPLE string is a structured prompt component used to instruct the Solver on how to react to specific rejection signals from the Verifier. This is vital for transferring the logic of the PCAF loop into the LLM's behavioral model, ensuring corrections are targeted.
The examples illustrate the three primary failure modes defined by the Verifier and the corresponding corrective action expected from the Solver:

## Solver Agent System Prompt Generation

This function dynamically generates the SOLVER\_ROLE system prompt, dictating the Solver agent's behavior and enforcing the framework's core requirement for verifiable, code-grounded solutions. We implemented a Dual-Verification Protocol: The central mechanism is the "ANALYTICAL vs NAIVE" CHECK. This mandates that the Solver must use two fundamentally different, cross-validating methods:
- Method 1 (Analytical): Symbolic and efficient algorithms (leveraging libraries like sympy).
- Method 2 (Naive Brute-Force): Simple, unoptimized simulation or counting loops.
This dual-approach ensures that if the Analytical method contains a subtle logical error, the Brute-Force "Dumb Check" will likely produce a conflicting answer, which the Verifier will then catch as a $\text{METHOD\_CONFLICT}$.

When a retry is requested ($\text{is\_retry=True}$), the function injects a strong ATTENTION warning. 

## Planner Agent: Deterministic Correction Routing

The construct\_planner\_feedback function is the implementation of the Planner agent. It performs the vital role of converting the Verifier's discrete JSON error type into a highly specific, prescriptive, and mandatory instruction for the Solver. This deterministic routing is the core mechanism that breaks the Solver's failure loop.

Logic (Deterministic Routing): The function uses a series of if/elif statements based on the $\text{error\_type}$ extracted from the Verifier's JSON output: 
- EXECUTION_FAILURE / RUNTIME ERROR: The instruction focuses solely on debugging the code syntax or outputting a value, delaying the math logic correction until the code runs.
- VALUE_MISMATCH (The Stubborn Fix): This triggers a Protocol Reset and Code-Grounding Enforcement. The Solver is told to IGNORE its previous text reasoning and RE-EXECUTE its code, accepting the code's output as the truth. This prevents the Solver from clinging to a numerically wrong answer.
- HARDCODING_SUSPICION (The Anti-Cheat Fix): The Solver is issued a METHODOLOGY REJECTION and forced to DERIVE the answer using a Python script (e.g., a loop for counting, $\text{sympy.solve}$ for equations), preventing it from outputting "magic numbers" or relying on recalled answers.
- CONCEPTUAL_FLAW / Geometry (The Strategy Shift): If a geometry-related term is found in the problem text, the Planner issues a STRATEGY SHIFT, commanding the Solver to abandon its previous flawed approach and switch to Coordinate Geometry. This forces a known-reliable, algorithmic method ($\text{Shoelace Formula}$, etc.), which is easier to verify.

In [ ]:
def construct_planner_feedback(verifier_json, problem_text, history_text=""):
    """
    Routes specific error types to 'Loop Breaking' instructions.
    """
    error_cat = verifier_json.get("error_type", "UNKNOWN") # Note: Key changed to error_type to match new prompt
    critique = verifier_json.get("critique", "")

    # 1. HANDLE RUNTIME ERRORS (Keep existing logic)
    last_turn_trace = history_text.split("--- Correction")[-1] if "--- Correction" in history_text else history_text
    if "RUNTIME ERROR" in last_turn_trace or error_cat == "EXECUTION_FAILURE":
        return (
            "The previous code attempt failed with a RUNTIME ERROR or produced NO OUTPUT. "
            "**MANDATORY ACTION:** Fix the syntax. Do not change the math logic yet; just make the code run. "
            "Print the final result explicitly."
        )

    # 2. HANDLE "VALUE_MISMATCH" (The "Stubborn Solver" Fix)
    # This fixes the issue in Problem 23 where Text=279 but Code=610
    if error_cat == "VALUE_MISMATCH":
        return (
            f"**CRITICAL CONFLICT:** The Verifier found a mismatch. {critique}. "
            "Your text claims one answer, but your Python code calculated a different one. "
            "**PROTOCOL:** \n"
            "1. IGNORE your previous text intuition. It was wrong.\n"
            "2. **RE-EXECUTE the code** to confirm the calculated value is stable.\n"
            "   Do NOT copy your previous reasoning. Write a COMPLETELY NEW derivation text that matches the new code."
            "3. If the code runs correctly, accept its output as the truth.\n"
            "**WARNING:** Do NOT just state the number from the feedback. You MUST run the code again to prove it."
        )

    # 3. HANDLE "HARDCODING_SUSPICION" (The "Anti-Cheat" Fix)
    # This prevents the "I recall..." or manual counting issues
    if error_cat == "HARDCODING_SUSPICION":
        return (
            "**METHODOLOGY REJECTION:** The Verifier flagged your answer as a 'Magic Number' or 'Manual Count'. "
            "**MANDATORY ACTION:** You must Delete your 'recalled' answer. Write a Python script to DERIVE the answer from scratch. "
            "If counting items, write a loop. If solving equations, use `sympy.solve`."
        )

    # 4. HANDLE "CONCEPTUAL_FLAW" (Geometry/Logic Routing)
    if error_cat == "CONCEPTUAL_FLAW":
        is_geometry = any(k in problem_text.lower() for k in ["geometry", "triangle", "polygon", "area"])
        if is_geometry:
             return (
                 "**STRATEGY SHIFT:** Your geometric reasoning was flagged as flawed. "
                 "**MANDATORY ACTION:** Switch to Coordinate Geometry. Place a vertex at (0,0), define all other points as (x,y) coordinates, and use the Shoelace Formula or Distance Formula in Python."
             )
        else:
            return (
                "**LOGIC ERROR:** Your mathematical approach is flawed. "
                "Step back. List the constraints again. Try a different method (e.g., if you used Algebra, try Brute Force with Python)."
            )

    # Fallback
    return f"The Verifier failed the solution. Reason: {critique}. Please fix the code to match the reasoning."

## Final Answer Extraction Logic

The extract\_answer function is a multi-strategy heuristic designed for robustly identifying and cleaning the final, scoreable answer from the Solver's output trace. This is crucial for automation because LLM output formats can be inconsistent.
- Strategy 1: $\text{\boxed{}}$ (Highest Priority): The function first scans for the $\text{\boxed{}}$ LaTeX command. The logic is specifically implemented to handle nested braces by using a counter ($\text{brace\_count}$), ensuring it accurately extracts the content even if the box contains complex LaTeX or internal grouping.
- Strategy 2: $\text{Final Answer}$ Marker: If no box is found, it searches for explicit textual markers like "Final Answer:" or "answer is:".Robust Token Extraction: Critically, it then applies a sub-strategy to extract only the numerical/mathematical token from the matched line. This prevents common errors where the final answer is $\text{18.897}$ but the Solver appends text like "cards)" or "is the result," which would corrupt the scoring.
- Strategy 3: Last Math Token (Fallback): As a final, low-confidence heuristic, the function extracts all mathematical tokens (integers, decimals, fractions, $\sqrt{}$, ${\pi}$) from the entire text and returns the last one found. This captures answers where the Solver fails to use any explicit marker.

## Evaluation Utility (Semantic Scoring):

The $\text{check\_corectness}$ function provides the final, robust layer of evaluation by ensuring that mathematically equivalent answers are scored correctly, regardless of their formatting (e.g., $\text{1/2}$ vs. $\text{0.5}$).
Logic (Numerical Equivalence): The core mechanism is a helper function that converts complex input strings (both the prediction and the ground truth) into Python-executable expressions.
- LaTeX Conversion: It uses regular expressions to convert raw LaTeX fractions (${\frac\{a\}\{b\}}$) into Python division syntax ${(a)/(b)}$, and maps math constants (like ${\sqrt{}}$ and ${\pi}$) to the Python $\text{math}$ library functions.
- Implied Multiplication Fix: Critically, it inserts a multiplication operator where implied multiplication exists (e.g., changing ${2\pi}$ to ${2*\pi}$), preventing common execution errors during evaluation.
Final Check: It uses the resulting numerical values to perform a check based on floating-point tolerance ($\text{1e-3}$), guaranteeing semantic correctness for non-integer answers.

## Data Bridge:



The $\text{extract\_json\_from\_response}$ function is a crucial data bridge that ensures the Verifier's LLM output (raw text critique) is reliably converted into a structured input for the Planner's deterministic logic. 
- Multi-Strategy Parsing: The function is built for extreme robustness by using sequential parsing attempts to handle different malformed outputs from the LLM:
    - It cleans up Markdown wrappers (e.g., $\text{```json}$).
    - It attempts standard JSON loading after fixing issues like trailing commas.
    - It falls back to $\text{ast.literal\_eval}$ for Python dictionary syntax (single quotes, $\text{True/False}$ booleans).
- Guaranteed Data Validation (Core Feature): The $\text{validate\_and\_default}$ helper is the most important part of this logic. After successful parsing, it checks if essential keys (like $\text{valid}$ and $\text{error\_type}$) are present. If they are missing, it injects safe default values ($\text{False}$ and $\text{NONE}$) to prevent the downstream Planner's deterministic routing logic from crashing due to unexpected schema holes.

## PCAF Orchestration Kernel

The $\text{run\_pcaf\_on\_problem}$ function implements the central control flow loop: Solve $\rightarrow$ Verify $\rightarrow$ Plan $\rightarrow$ (Re-)Solve. It manages the multi-agent state to enable iterative self-correction, limited by a hard constraint.
Core Logic (The Constrained Loop): 
- The system is designed to execute a maximum of four attempts (one initial attempt + $\text{max\_retries}=3$ corrections). The loop controls the sequencing of the three agents (Solver, Verifier, Planner) until a valid solution is found or the limit is reached.
- Correction Enforcement: Each correction step ensures targeted debugging by passing the full history of failures and a prescriptive $\text{planner\_instruction}$ back to the Solver. If $\text{is\_retry=True}$, the Solver is commanded to perform a Protocol Reset ("Wipe your memory...") to prevent it from repeating the past mistake.
- State Management: The loop persists the single, stateful $\text{PersistentSolverSandbox}$ across all four attempts, allowing complex multi-step code workflows to continue even after a correction/retry.

In [ ]:
def run_pcaf_on_problem(problem_text, max_retries=3, temperature=0.7):
    all_tries = []
    problem_sandbox = PersistentSolverSandbox()
    
    # 1. Initial Attempt
    current_solution, trace = run_solver_with_tools(
        get_solver_prompt(is_retry=False), 
        problem_text,
        problem_sandbox,
        temperature=temperature
    )
    
    # Initialize loop variables
    # We treat the initial attempt as "Attempt 0"
    structured_history = [f"=== ATTEMPT 1 (ORIGINAL) ===\n{trace}"]
    
    # We loop up to max_retries + 1 because we want to Verify the INITIAL attempt too.
    # The loop condition controls how many CORRECTIONS we allow.
    for attempt_idx in range(max_retries + 1):
        
        # --- A. VERIFY CURRENT SOLUTION ---
        print(f"--- Verifying Attempt {attempt_idx + 1} ---")
        
        # Extract evidence from the CURRENT trace
        code_outputs = re.findall(r"\[Tool Output\]\s*(.*?)(?=\n\n--- Step|\Z)", trace, re.DOTALL)
        if not code_outputs:
            evidence_str = "NO CODE EXECUTED."
        else:
            evidence_str = ""
            for idx, out in enumerate(code_outputs):
                evidence_str += f"=== EXECUTION {idx+1} ===\n{out.strip()}\n\n"

        verifier_input = (
            f"Problem: {problem_text}\n\n"
            f"=== EVIDENCE (CODE OUTPUTS) ===\n{evidence_str}\n\n"
            f"=== SOLVER'S REASONING ===\n{trace}"
        )
        
        verifier_raw = get_llm_response([
            {"role": "system", "content": VERIFIER_ROLE},
            {"role": "user", "content": verifier_input}
        ])
        verifier_json = extract_json_from_response(verifier_raw)
        
        # Log this attempt
        current_try = {
            "turn": attempt_idx,
            "solution_text": current_solution,
            "full_trace": trace,
            "verifier_json": verifier_json,
            "planner_instruction": None
        }
        all_tries.append(current_try)
        
        # --- B. CHECK SUCCESS ---
        if verifier_json and verifier_json.get("valid", False):
            print(f"Verifier Verdict: VALID (Attempts: {attempt_idx + 1})")
            return current_solution, all_tries
        
        print(f"Verifier Verdict: INVALID ({verifier_json.get('error_type')})")
        
        # --- C. STOP IF OUT OF RETRIES ---
        if attempt_idx == max_retries:
            print("Max retries reached. Stopping.")
            break
            
        # --- D. PLAN & CORRECT (If we have retries left) ---
        planner_instruction = construct_planner_feedback(verifier_json, problem_text, trace)
        all_tries[-1]["planner_instruction"] = planner_instruction # Log instruction
        
        print(f"--- Generating Correction {attempt_idx + 1} ---")
        history_text = "\n\n".join(structured_history)
        
        solver_input_context = (
            f"ORIGINAL PROBLEM: {problem_text}\n\n"
            f"--- HISTORY OF FAILURES ---\n{history_text}\n\n"
            f"--- FEEDBACK ---\n"
            f"VERIFIER: {verifier_json.get('critique', 'Invalid')}\n"
            f"INSTRUCTION: {planner_instruction}\n"
        )
        
        current_solution, new_trace = run_solver_with_tools(
            get_solver_prompt(is_retry=True),
            solver_input_context,
            problem_sandbox,
            temperature=temperature
        )
        
        # Update trace for the next loop iteration
        trace = new_trace 
        structured_history.append(f"=== ATTEMPT {attempt_idx + 2} (CORRECTION) ===\n{new_trace}")

    return current_solution, all_tries

## Evaluation Protocol:

This function applies the MathArena evaluation protocol to the PCAF system, testing not just correctness, but reliability across multiple independent runs.
- Protocol (Multiple Independent Runs): The core mechanism is a double loop. For every problem, the entire $\text{run\_pcaf\_on\_problem}$ cycle (the full Solve-Verify-Plan loop) is executed four times ($\text{attempts\_per\_problem=4}$). Each of these four runs is a separate, fresh, and independent evaluation of the PCAF.
- Final Metric (Averaged Accuracy): The score for a single problem is not binary ($\text{0}$ or $\text{1}$), but the average accuracy across the four runs. For instance, if the PCAF solves the problem correctly in 3 out of 4 independent runs, the problem's score is $\text{0.75}$. This protocol accounts for the inherent stochasticity of LLMs (due to $\text{temperature=0.7}$) and provides a much more robust measure of system performance.
- Data Logging: It diligently logs the results from all four runs, including the answers, the scores, and the full histories ($\text{run\_histories}$), providing the necessary data to analyze system stability and self-correction behavior under variance.

In [ ]:
def run_matharena_pcaf_benchmark(dataframe, num_samples=None, attempts_per_problem=4, temperature=0.7):
    """
    Evaluates the PCAF System using MathArena protocol:
    - 4 independent PCAF runs per problem.
    - Score is average accuracy of the 4 final PCAF outputs.
    """
    subset = dataframe.head(num_samples).copy() if num_samples else dataframe.copy()
    results = []

    print(f"=======================================================")
    print(f"STARTING MATHARENA PCAF EVALUATION")
    print(f"Problems: {len(subset)} | PCAF Runs/Prob: {attempts_per_problem} | Solver Temp: {temperature}")
    print(f"=======================================================")

    for index, row in subset.iterrows():
        problem_id = row['problem_idx']
        ground_truth = str(row['answer'])
        problem_text = row['problem']
        
        print(f"\n--- Problem ID: {problem_id} ---")
        
        run_scores = []
        run_answers = []
        run_turns = []
        run_histories = []
        
        for i in range(attempts_per_problem):
            # Run the FULL PCAF LOOP (Solver+Verifier+Planner)
            # This counts as ONE attempt
            final_sol, history = run_pcaf_on_problem(
                problem_text, 
                max_retries=3, 
                temperature=temperature
            )
            
            # Extract final answer from the result of the PCAF loop
            extracted_val = extract_answer(final_sol)
            is_correct = check_correctness(extracted_val, ground_truth)
            
            score = 1 if is_correct else 0
            run_scores.append(score)
            run_answers.append(extracted_val)
            run_turns.append(len(history))
            run_histories.append(history)
            
            # Log result
            mark = "True" if is_correct else "False"
            # We track how many internal turns PCAF took (e.g., did it self-correct?)
            print(f"   PCAF Run {i+1}: {mark} (Ans: {extracted_val}) [Internal Turns: {len(history)}]")
            
        # Calc Stats
        avg_score = sum(run_scores) / attempts_per_problem
        print(f"   >> Avg PCAF Accuracy: {avg_score:.2f}")
        
        results.append({
            "problem_idx": problem_id,
            "ground_truth": ground_truth,
            "pcaf_answers": run_answers,
            "pcaf_scores": run_scores,
            "pcaf_turns": run_turns,
            "final_score": avg_score,
            # Serialize history to JSON so it saves cleanly to CSV
            "pcaf_full_histories": [json.dumps(h) for h in run_histories]
        })
        
    results_df = pd.DataFrame(results)
    print(f"\nGlobal PCAF Accuracy: {results_df['final_score'].mean() * 100:.2f}%")
    return results_df

# --- EXECUTION --- 
competitions = (aime_df, hmmt_feb_df)
for index, comp_df in enumerate(competitions):
    pcaf_matharena_results = run_matharena_pcaf_benchmark(comp_df, num_samples=1)
    pcaf_matharena_results.to_csv(f"PCAF_Final_Result_{index}.csv", index=False) 
    print(f"\nDetailed results saved to PCAF_Final_Result_{index}.csv")